# Notebook 12: Failure-Aware Hybrid Model & Ablation Study
This notebook designs a Failure-Aware Hybrid Model and evaluates it progressively against standard baselines without any test-set data leakage.


In [1]:
import sys, os, time
sys.path.insert(0, os.path.abspath(".."))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from src.config import *
from src.data_utils import load_artifact, save_artifact
from src.evaluation import compute_all_metrics, metrics_to_dataframe

seed_everything()

# Load indices
train_idx = np.load(os.path.join(DATA_PROCESSED, "train_indices.npy"))
test_idx = np.load(os.path.join(DATA_PROCESSED, "test_indices.npy"))

# Load true labels
df = pd.read_csv(os.path.join(DATA_PROCESSED, "df1_engineered.csv"))
y_train = df.loc[train_idx, TARGET_COL].values
y_test = df.loc[test_idx, TARGET_COL].values

# Load OOF Predictions (No Leakage)
oof_df = pd.read_csv(os.path.join(RESULTS_METRICS, "09_tuned_oof_predictions.csv"))
test_preds_df = pd.read_csv(os.path.join(RESULTS_METRICS, "09_tuned_test_predictions.csv"))

models = ["XGBoost", "LightGBM", "CatBoost", "RandomForest", "LogisticRegression", "DecisionTree", "SVM"]
# Remove models that failed or don't exist
models = [m for m in models if f"{m}_pred" in oof_df.columns]

print("Available models:", models)


Available models: ['XGBoost', 'LightGBM', 'CatBoost', 'RandomForest', 'LogisticRegression', 'DecisionTree', 'SVM']


### 1. Constructing Failure-Aware Meta-Features

In [2]:
def extract_meta_features(df, models, n_classes=3):
    meta_X = []
    feature_names = []
    
    # 1. Base Probabilities (Standard Stacking)
    probs = []
    for m in models:
        for c in range(n_classes):
            probs.append(df[f"{m}_prob_{c}"].values)
            if len(meta_X) == 0:
                feature_names.append(f"{m}_prob_{c}")
    meta_X = np.column_stack(probs)
    
    # 2. Maximum Confidence (Entropy / Uncertainty)
    conf = []
    for m in models:
        model_probs = np.column_stack([df[f"{m}_prob_{c}"].values for c in range(n_classes)])
        max_conf = np.max(model_probs, axis=1)
        entropy = -np.sum(model_probs * np.log(model_probs + 1e-9), axis=1)
        margin = max_conf - np.sort(model_probs, axis=1)[:, -2]
        
        conf.extend([max_conf, entropy, margin])
        if len(feature_names) == meta_X.shape[1]:
            feature_names.extend([f"{m}_max_conf", f"{m}_entropy", f"{m}_margin"])
    
    conf_X = np.column_stack(conf)
    
    # 3. Disagreement
    preds = np.column_stack([df[f"{m}_pred"].values for m in models])
    mode_pred = np.apply_along_axis(lambda x: np.bincount(x, minlength=n_classes).argmax(), axis=1, arr=preds)
    
    disagreements = []
    for i, m in enumerate(models):
        disagreements.append((preds[:, i] != mode_pred).astype(int))
        if len(feature_names) == meta_X.shape[1] + conf_X.shape[1]:
            feature_names.append(f"{m}_disagrees_with_consensus")
            
    # Agreement count
    agreement_count = np.apply_along_axis(lambda x: np.max(np.bincount(x, minlength=n_classes)), axis=1, arr=preds)
    disagreements.append(agreement_count)
    if len(feature_names) == meta_X.shape[1] + conf_X.shape[1] + len(models):
        feature_names.append("consensus_agreement_count")
        
    dis_X = np.column_stack(disagreements)
    
    return meta_X, conf_X, dis_X, feature_names

# Train Data
meta_X_train, conf_X_train, dis_X_train, meta_cols = extract_meta_features(oof_df, models, N_CLASSES)

# Test Data (Locked, only for final inference)
meta_X_test, conf_X_test, dis_X_test, _ = extract_meta_features(test_preds_df, models, N_CLASSES)


### 2. Ablation Study Models
We progressively evaluate:
A. Best single base model
B. Simple probability averaging
C. Standard stacking (Base probabilities only)
E. Base probabilities + confidence
F. Base probabilities + disagreement
G. Base probabilities + confidence + disagreement
H. Full proposed failure-aware hybrid


In [3]:
ablation_results = []
ablations = {
    "A. Best Single Model (XGBoost)": None,
    "B. Soft Voting Baseline": None,
    "C. Standard Stacking (Probs Only)": meta_X_train,
    "E. Probs + Confidence": np.hstack([meta_X_train, conf_X_train]),
    "F. Probs + Disagreement": np.hstack([meta_X_train, dis_X_train]),
    "G. Probs + Conf + Disagreement (Failure-Aware)": np.hstack([meta_X_train, conf_X_train, dis_X_train]),
}

ablations_test = {
    "C. Standard Stacking (Probs Only)": meta_X_test,
    "E. Probs + Confidence": np.hstack([meta_X_test, conf_X_test]),
    "F. Probs + Disagreement": np.hstack([meta_X_test, dis_X_test]),
    "G. Probs + Conf + Disagreement (Failure-Aware)": np.hstack([meta_X_test, conf_X_test, dis_X_test]),
}

final_hybrid_model = None

for name, X_tr in ablations.items():
    print(f"\nEvaluating: {name}")
    
    if name == "A. Best Single Model (XGBoost)":
        y_prob = np.column_stack([test_preds_df[f"XGBoost_prob_{c}"] for c in range(N_CLASSES)])
        y_pred = test_preds_df["XGBoost_pred"].values
    elif name == "B. Soft Voting Baseline":
        y_prob = np.zeros((len(test_preds_df), N_CLASSES))
        for m in models:
            y_prob += np.column_stack([test_preds_df[f"{m}_prob_{c}"] for c in range(N_CLASSES)])
        y_prob /= len(models)
        y_pred = np.argmax(y_prob, axis=1)
    else:
        # Meta-learner: Logistic Regression (interpretable, regularized)
        meta_learner = LogisticRegression(class_weight="balanced", random_state=RANDOM_SEED, max_iter=1000)
        meta_learner.fit(X_tr, y_train)
        
        X_te = ablations_test[name]
        y_prob = meta_learner.predict_proba(X_te)
        y_pred = meta_learner.predict(X_te)
        
        if "Failure-Aware" in name:
            final_hybrid_model = meta_learner

    metrics = compute_all_metrics(y_test, y_pred, y_prob, model_name=name)
    ablation_results.append(metrics)
    print(f"  Macro F1: {metrics['Macro_F1']:.4f} | Balanced Acc: {metrics['Balanced_Accuracy']:.4f}")

ablation_df = metrics_to_dataframe(ablation_results)
ablation_df.to_csv(os.path.join(RESULTS_METRICS, "12_hybrid_ablation_results.csv"), index=False)
print("\nSaved 12_hybrid_ablation_results.csv")
display(ablation_df)



Evaluating: A. Best Single Model (XGBoost)
  Macro F1: 0.4347 | Balanced Acc: 0.4689

Evaluating: B. Soft Voting Baseline


  Macro F1: 0.4476 | Balanced Acc: 0.5028

Evaluating: C. Standard Stacking (Probs Only)


  Macro F1: 0.4199 | Balanced Acc: 0.5260

Evaluating: E. Probs + Confidence


  Macro F1: 0.4230 | Balanced Acc: 0.5222

Evaluating: F. Probs + Disagreement


  Macro F1: 0.4197 | Balanced Acc: 0.5294

Evaluating: G. Probs + Conf + Disagreement (Failure-Aware)


  Macro F1: 0.4241 | Balanced Acc: 0.5232

Saved 12_hybrid_ablation_results.csv


,Accuracy,Macro_Precision,Macro_Recall,Macro_F1,Balanced_Accuracy,MCC,Precision_Class0,Recall_Class0,F1_Class0,Precision_Class1,Recall_Class1,F1_Class1,Precision_Class2,Recall_Class2,F1_Class2,ROC_AUC_OVR,PR_AUC,Log_Loss,Brier_Score
Model,,,,,,,,,,,,,,,,,,,
A. Best Single Model (XGBoost),0.758081,0.424047,0.468876,0.434679,0.468876,0.308014,0.913400,0.802999,0.854649,0.017658,0.020518,0.018981,0.341084,0.583109,0.430406,0.732031,0.452451,0.549170,0.107352
B. Soft Voting Baseline,0.747674,0.437575,0.502774,0.447611,0.502774,0.353318,0.933443,0.771765,0.844939,0.037143,0.042117,0.039474,0.342138,0.694441,0.458421,0.767613,0.467758,0.671301,0.124264
C. Standard Stacking (Probs Only),0.611716,0.449860,0.526027,0.419926,0.526027,0.294614,0.959784,0.615896,0.750314,0.028259,0.340173,0.052183,0.361536,0.622012,0.457283,0.771165,0.469862,0.883230,0.164892
E. Probs + Confidence,0.639231,0.440724,0.522238,0.422991,0.522238,0.299838,0.953954,0.646124,0.770428,0.030842,0.275378,0.055471,0.337377,0.645211,0.443074,0.767274,0.467675,0.861522,0.161157
F. Probs + Disagreement,0.611676,0.449520,0.529405,0.419748,0.529405,0.293220,0.959305,0.616621,0.750705,0.029535,0.356371,0.054550,0.359719,0.615221,0.453990,0.769769,0.468737,0.876949,0.163845
G. Probs + Conf + Disagreement (Failure-Aware),0.638738,0.443181,0.523185,0.424149,0.523185,0.298312,0.953877,0.648394,0.772014,0.030919,0.295896,0.055987,0.344747,0.625265,0.444444,0.767345,0.466861,0.856897,0.160507


In [4]:
# Save the final hybrid model and metadata
save_artifact(final_hybrid_model, os.path.join(MODELS_HYBRID, "hybrid_meta_model.joblib"))

hybrid_metadata = {
    "base_models": models,
    "meta_features": meta_cols,
    "n_classes": N_CLASSES
}
save_artifact(hybrid_metadata, os.path.join(MODELS_HYBRID, "hybrid_metadata.joblib"))

print("Saved production artifacts for the dashboard.")


Saved: d:\DAV\AI_Driven_Public_Health_Surveillance\models/hybrid\hybrid_meta_model.joblib
Saved: d:\DAV\AI_Driven_Public_Health_Surveillance\models/hybrid\hybrid_metadata.joblib
Saved production artifacts for the dashboard.
